In [ ]:
import torch
import torch.nn as nn
import hparams as hp
from model import MelNet
from torch.utils.data import DataLoader
from data import MelData, MelCollate
import matplotlib.pyplot as plt
from utils import *
import numpy as np
%matplotlib inline
np.set_printoptions(precision=3, suppress=True)
dataset = MelData(hp)
collate_fn = MelCollate(hp)
dataloader = DataLoader(dataset, batch_size = hp.batch_size, shuffle=True, drop_last=True, collate_fn = collate_fn)

In [ ]:
device = torch.device('cuda:1')
model = MelNet(hp)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)
criterion = nn.L1Loss()

print("Training Start!!!")
iteration=1
for epoch in range(hp.epochs):
    for tiers in dataloader:
        lr_scheduling(optimizer, iteration, 3e-4, 10)
        
        model.zero_grad()
        target = tiers[1].to(device)
        y_pred = model(target)

        loss = criterion(y_pred, target)
        loss.backward()
        optimizer.step()
        
        iteration += 1
        
        if iteration%20==0:
            fig = plt.figure(figsize=(20,10))
            ax1, ax2, ax3 = fig.add_subplot(221), fig.add_subplot(222), fig.add_subplot(223)
            ax1.imshow(target[0].cpu().numpy(), origin='lower')
            ax2.imshow(y_pred[0].cpu().detach().numpy(), origin='lower')
            ax3.imshow(np.tile(y_pred[0][0:1].cpu().detach().numpy(), (16,1)), origin='lower')
            plt.show()
            print(y_pred[0][0].cpu().detach().numpy()[:10])
            print(y_pred[0][1].cpu().detach().numpy()[:10])
        print(loss.item())
    break

In [ ]:
plt.imshow( y_pred[0].cpu().detach().numpy(), origin='lower')

In [ ]:
y_pred[0][2]